In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
import datetime as dt

C:\Users\aengland\Anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\aengland\Anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2025-01-08 17:23:20.092155


### Functions

In [3]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [4]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# output
str_dirname_output = './output'

Project: 20241112-simple-model-test
Task: 02_get_tsp_data


### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Read query

In [6]:
str_filepath = './sql/query.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('with tblMax as\n'
 '(\n'
 'select\n'
 '\tbigAccountId,\n'
 '\tmax(concat(MonthOnBooks, bigAccountid, bigRunDateKeyId)) as MaxUnique\n'
 'from riskdb.dbo.tblReportCOStaticPools_StaticPool\n'
 'where MonthOnBooks<=72\n'
 'group by bigAccountId\n'
 '),\n'
 '\n'
 'tblReportCOStaticPools_StaticPoolNew as\n'
 '(\n'
 'select\n'
 '\tconcat(MonthOnBooks, bigAccountid, bigRunDateKeyId) as uniqueC,\n'
 '\t*\n'
 'from riskdb.dbo.tblReportCOStaticPools_StaticPool\n'
 'where MonthOnBooks<=72\n'
 '),\n'
 '\n'
 'debt AS\n'
 '(\n'
 '\tSELECT\n'
 '\t\tbigAccountId,\n'
 '\t\tbigDebtorId,\n'
 '\t\tISNULL(SUM(fltMonthlyPayment), 0) AS debt \n'
 '\tFROM electra.pfsdb.dbo.tblDebt d\n'
 '\tWHERE bitUse = 1 AND fltMonthlyPayment IS NOT NULL\n'
 '\tGROUP BY bigAccountId, bigDebtorId\n'
 ')\n'
 '\n'
 '\n'
 'select \n'
 '\tCONCAT(tblAccount.bigAccountId, tblAccount.bigDebtorId, 1) as UniqueID,\n'
 '\ttbltempstaticpool.bigAccountId,\n'
 '\ttblAccount.bigDebtorId,\n'
 '\t1 as bitDebtor,\n'
 '\ttblAccount.dtmStamp

### Write into df

In [7]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# show
df

<timed exec>:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


Wall time: 5min 38s


,UniqueID,bigAccountId,bigDebtorId,bitDebtor,dtmStampCreation,dtmFunded,bitDefault,MonthOnBooks,RunningNetLoss,AmtFinanced,...,fltApprovedDownTotal,Payment,DTI,PTI,bitServiceContract,fltAdvance,strVehicleType,bitGap,DealerStampCreation,totaldebt
0,546593468846511,5465934,6884651,1,2021-01-01 09:41:26.617,2021-01-07,0,48,0.0,15779.71,...,0.0,362.51,0.249783,0.057318,0,1.192262,auto,1,2020-10-23 13:11:54.257,1091.00
1,546599768847241,5465997,6884724,1,2021-01-01 11:13:07.147,2021-01-11,0,48,0.0,16545.58,...,2160.0,381.96,0.327537,0.119892,1,0.898571,auto,1,2020-02-07 08:29:35.927,661.53
2,546604668847831,5466046,6884783,1,2021-01-01 12:26:26.883,2021-01-18,0,48,0.0,24572.00,...,0.0,556.62,0.383475,0.113944,1,1.118690,suv,0,2014-03-14 16:04:13.850,1316.67
3,546611768848671,5466117,6884867,1,2021-01-01 13:40:20.980,2021-01-20,0,48,0.0,16239.90,...,4000.0,384.75,0.377446,0.091937,1,1.104110,van,1,2020-06-17 10:05:47.617,1194.83
4,546611868848681,5466118,6884868,1,2021-01-01 13:41:09.923,2021-01-14,0,48,0.0,22420.92,...,0.0,508.41,0.464816,0.081186,1,1.125723,suv,1,2017-04-07 09:23:49.460,1850.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101948,8535032105278331,8535032,10527834,0,2024-12-31 16:04:16.250,2025-01-02,0,None,NaN,34991.50,...,0.0,1022.20,0.368418,0.092049,0,1.133690,suv,1,2022-03-04 11:38:51.370,19.86
101949,8536877105299991,8536877,10530000,0,2025-01-02 11:38:41.897,2025-01-07,0,None,NaN,30840.40,...,0.0,825.27,0.450806,0.122711,0,1.035246,auto,1,2022-08-19 14:38:01.340,836.48
101950,8538151105315191,8538151,10531520,0,2025-01-02 16:03:32.067,2025-01-06,0,None,NaN,16089.34,...,500.0,463.58,0.279084,0.079084,1,1.103788,suv,1,2022-03-29 16:06:09.463,NaN
101951,8543705105382391,8543705,10538240,0,2025-01-04 14:39:02.697,2025-01-07,0,None,NaN,18339.08,...,5200.0,538.27,0.499661,0.115027,0,0.993146,auto,1,2022-11-28 15:22:35.180,864.00


### Lower column names and suffix

In [8]:
# lower columns
list_cols = [f'{col.lower()}__app' for col in df.columns]
df.columns = list_cols
# show
df

,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,dtmstampcreation__app,dtmfunded__app,bitdefault__app,monthonbooks__app,runningnetloss__app,amtfinanced__app,...,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app,totaldebt__app
0,546593468846511,5465934,6884651,1,2021-01-01 09:41:26.617,2021-01-07,0,48,0.0,15779.71,...,0.0,362.51,0.249783,0.057318,0,1.192262,auto,1,2020-10-23 13:11:54.257,1091.00
1,546599768847241,5465997,6884724,1,2021-01-01 11:13:07.147,2021-01-11,0,48,0.0,16545.58,...,2160.0,381.96,0.327537,0.119892,1,0.898571,auto,1,2020-02-07 08:29:35.927,661.53
2,546604668847831,5466046,6884783,1,2021-01-01 12:26:26.883,2021-01-18,0,48,0.0,24572.00,...,0.0,556.62,0.383475,0.113944,1,1.118690,suv,0,2014-03-14 16:04:13.850,1316.67
3,546611768848671,5466117,6884867,1,2021-01-01 13:40:20.980,2021-01-20,0,48,0.0,16239.90,...,4000.0,384.75,0.377446,0.091937,1,1.104110,van,1,2020-06-17 10:05:47.617,1194.83
4,546611868848681,5466118,6884868,1,2021-01-01 13:41:09.923,2021-01-14,0,48,0.0,22420.92,...,0.0,508.41,0.464816,0.081186,1,1.125723,suv,1,2017-04-07 09:23:49.460,1850.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101948,8535032105278331,8535032,10527834,0,2024-12-31 16:04:16.250,2025-01-02,0,None,NaN,34991.50,...,0.0,1022.20,0.368418,0.092049,0,1.133690,suv,1,2022-03-04 11:38:51.370,19.86
101949,8536877105299991,8536877,10530000,0,2025-01-02 11:38:41.897,2025-01-07,0,None,NaN,30840.40,...,0.0,825.27,0.450806,0.122711,0,1.035246,auto,1,2022-08-19 14:38:01.340,836.48
101950,8538151105315191,8538151,10531520,0,2025-01-02 16:03:32.067,2025-01-06,0,None,NaN,16089.34,...,500.0,463.58,0.279084,0.079084,1,1.103788,suv,1,2022-03-29 16:06:09.463,NaN
101951,8543705105382391,8543705,10538240,0,2025-01-04 14:39:02.697,2025-01-07,0,None,NaN,18339.08,...,5200.0,538.27,0.499661,0.115027,0,0.993146,auto,1,2022-11-28 15:22:35.180,864.00


### Save as parquet

In [9]:
%%time

# save
str_filename = 'df_tsp.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_parquet(str_local_path, compression='gzip')

Wall time: 2.01 s


### Upload to s3

In [10]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_filename}', 
    str_bucket_name=str_project,
)

Wall time: 512 ms


### Clean-up

In [11]:
os.remove(str_local_path)